# 🩺 Smart Diabetes Monitoring and Prediction System
## Notebook 5 — Model Selection, Hyperparameter Tuning, and Threshold Optimization

Selects best model via composite scoring, tunes hyperparameters with RandomizedSearchCV, and identifies clinical deployment threshold (Recall ≥ 0.90).

### Setup

In [7]:
import os, sys, time, warnings, pickle, json
from pathlib import Path
from datetime import datetime

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score,
    RandomizedSearchCV, learning_curve
)
from sklearn.preprocessing import RobustScaler, label_binarize
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    AdaBoostClassifier, ExtraTreesClassifier
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report,
    precision_recall_curve, average_precision_score,
    matthews_corrcoef, balanced_accuracy_score, log_loss
)
from sklearn.feature_selection import mutual_info_classif
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from scipy.stats import pointbiserialr

try:
    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline import Pipeline as ImbPipeline
    SMOTE_AVAILABLE = True
    print("✓ imbalanced-learn available")
except ImportError:
    SMOTE_AVAILABLE = False
    print("✗ imbalanced-learn not installed")

try:
    import xgboost as xgb
    XGB_AVAILABLE = True
    print("✓ XGBoost available")
except ImportError:
    XGB_AVAILABLE = False

try:
    import lightgbm as lgb
    LGB_AVAILABLE = True
    print("✓ LightGBM available")
except ImportError:
    LGB_AVAILABLE = False

try:
    import catboost as cb
    CAT_AVAILABLE = True
    print("✓ CatBoost available")
except ImportError:
    CAT_AVAILABLE = False

print("\n✅ All libraries loaded.")

✓ imbalanced-learn available
✓ XGBoost available
✓ LightGBM available
✓ CatBoost available

✅ All libraries loaded.


In [ ]:
# ── UPDATE THIS PATH TO YOUR DATASET LOCATION ────────────────────────
DATA_PATH   = r"C:\Users\Ahmed A. Almansour\Documents\Final_Project _Submission\Graduation_Project_Files\Dataset\diabetes_binary_health_indicators_BRFSS2015.csv"
OUTPUT_ROOT = Path(r"C:\Users\Ahmed A. Almansour\Documents\Final_Project _Submission\Graduation_Project_Files\Outputs_Folder")

DIRS = {
    "models"       : OUTPUT_ROOT / "models",
    "scalers"      : OUTPUT_ROOT / "scalers",
    "plots_eda"    : OUTPUT_ROOT / "plots" / "01_eda",
    "plots_imb"    : OUTPUT_ROOT / "plots" / "02_imbalance",
    "plots_pre"    : OUTPUT_ROOT / "plots" / "03_preprocessing",
    "plots_base"   : OUTPUT_ROOT / "plots" / "04_baseline_models",
    "plots_eng"    : OUTPUT_ROOT / "plots" / "05_feature_engineering",
    "plots_eng_m"  : OUTPUT_ROOT / "plots" / "06_engineered_models",
    "plots_fi"     : OUTPUT_ROOT / "plots" / "07_feature_importance",
    "plots_sel"    : OUTPUT_ROOT / "plots" / "08_selected_models",
    "plots_cmp"    : OUTPUT_ROOT / "plots" / "09_model_comparison",
    "plots_best"   : OUTPUT_ROOT / "plots" / "10_best_model",
    "reports"      : OUTPUT_ROOT / "reports",
    "features"     : OUTPUT_ROOT / "feature_lists",
    "thresholds"   : OUTPUT_ROOT / "thresholds",
    "pipeline"     : OUTPUT_ROOT / "pipeline",
}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

print(f"✅ Output root: {OUTPUT_ROOT.resolve()}")

✅ Output root: C:\ProjectNoteBooks\Notebooks_Project\Final_Project_Files\Outputs_Folder


In [9]:
PALETTE   = ["#00B4D8","#0077B6","#48CAE4","#90E0EF",
             "#ADE8F4","#023E8A","#03045E","#CAF0F8"]
COLOR_POS = "#EF233C"
COLOR_NEG = "#00B4D8"
COLOR_ACC = "#2EC4B6"
COLOR_AUC = "#FF9F1C"
COLOR_ROC = "#0077B6"
COLOR_PR  = "#2EC4B6"
COLOR_F1  = "#FF9F1C"

sns.set_style("whitegrid")
sns.set_palette(PALETTE)
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor"  : "#F8FBFF",
    "axes.edgecolor"  : "#CCCCCC",
    "axes.titlesize"  : 14,
    "axes.labelsize"  : 12,
    "xtick.labelsize" : 10,
    "ytick.labelsize" : 10,
    "font.family"     : "DejaVu Sans",
})

SEED     = 42
CV_FOLDS = 5
TOP_N    = 15
np.random.seed(SEED)
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)

all_results = {}

print(f"✅ SEED={SEED}  CV_FOLDS={CV_FOLDS}  TOP_N={TOP_N}")

✅ SEED=42  CV_FOLDS=5  TOP_N=15


In [10]:
def save_pkl(obj, path, label=""):
    with open(path, "wb") as f:
        pickle.dump(obj, f)
    print(f"  [SAVED] {label} → {path}")

def load_pkl(path):
    with open(path, "rb") as f:
        return pickle.load(f)

def savefig(fig, path, dpi=150):
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    print(f"  [PLOT]  {path.name}")

def section(title):
    print("\n" + "━"*70)
    print(f"  {title}")
    print("━"*70)

def print_classification_report(y_true, y_pred, model_name, phase=""):
    print(f"\n{'─'*60}")
    print(f"  📊 Classification Report — {model_name}")
    if phase:
        print(f"     Phase: {phase}")
    print(f"{'─'*60}")
    print(classification_report(
        y_true, y_pred,
        target_names=["No Diabetes (0)", "Diabetes (1)"],
        digits=4))
    print(f"{'─'*60}")

def evaluate_model(model, X_tr, y_tr, X_te, y_te, name,
                   cv_obj=None, phase="", verbose=True):
    t0 = time.time()
    model.fit(X_tr, y_tr)
    elapsed = time.time() - t0
    y_pred  = model.predict(X_te)
    y_proba = (model.predict_proba(X_te)[:, 1]
               if hasattr(model, "predict_proba") else None)
    metrics = dict(
        accuracy      = accuracy_score(y_te, y_pred),
        precision     = precision_score(y_te, y_pred, zero_division=0),
        recall        = recall_score(y_te, y_pred, zero_division=0),
        f1            = f1_score(y_te, y_pred, zero_division=0),
        balanced_acc  = balanced_accuracy_score(y_te, y_pred),
        mcc           = matthews_corrcoef(y_te, y_pred),
        roc_auc       = roc_auc_score(y_te, y_proba) if y_proba is not None else np.nan,
        avg_precision = average_precision_score(y_te, y_proba) if y_proba is not None else np.nan,
        log_loss_val  = log_loss(y_te, y_proba) if y_proba is not None else np.nan,
        train_time_s  = elapsed,
        cv_roc_auc    = np.nan,
    )
    if cv_obj is not None:
        scores = cross_val_score(model, X_tr, y_tr,
                                 cv=cv_obj, scoring="roc_auc", n_jobs=-1)
        metrics["cv_roc_auc"] = scores.mean()
    if verbose:
        print(f"  {name:<35} Acc={metrics['accuracy']:.4f}  "
              f"F1={metrics['f1']:.4f}  Recall={metrics['recall']:.4f}  "
              f"Prec={metrics['precision']:.4f}  AUC={metrics['roc_auc']:.4f}  "
              f"t={elapsed:.1f}s")
        print_classification_report(y_te, y_pred, name, phase)
    return metrics

def plot_roc_pr(y_true, models_proba, title_prefix, save_dir):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for name, proba in models_proba.items():
        fpr, tpr, _ = roc_curve(y_true, proba)
        auc = roc_auc_score(y_true, proba)
        axes[0].plot(fpr, tpr, lw=2, label=f"{name}  AUC={auc:.3f}")
        prec, rec, _ = precision_recall_curve(y_true, proba)
        ap = average_precision_score(y_true, proba)
        axes[1].plot(rec, prec, lw=2, label=f"{name}  AP={ap:.3f}")
    axes[0].plot([0,1],[0,1],"k--",lw=1,alpha=.5)
    axes[0].set(xlabel="FPR", ylabel="TPR", title=f"{title_prefix} — ROC Curve")
    axes[0].legend(fontsize=7)
    baseline_ap = y_true.mean()
    axes[1].axhline(baseline_ap, color="gray", linestyle="--",
                    label=f"Baseline AP={baseline_ap:.3f}")
    axes[1].set(xlabel="Recall", ylabel="Precision",
                title=f"{title_prefix} — Precision-Recall Curve")
    axes[1].legend(fontsize=7)
    fig.tight_layout()
    savefig(fig, save_dir / f"{title_prefix.replace(' ','_')}_roc_pr.png")
    plt.show()

def results_to_df(results_dict):
    rows = []
    for phase, models in results_dict.items():
        for mname, m in models.items():
            row = {"Phase": phase, "Model": mname}
            row.update(m)
            rows.append(row)
    return pd.DataFrame(rows)

def get_models():
    m = {
        "Logistic Regression": LogisticRegression(
            max_iter=1000, random_state=SEED, class_weight="balanced"),
        "Random Forest"      : RandomForestClassifier(
            n_estimators=200, n_jobs=-1, random_state=SEED,
            class_weight="balanced"),
        "Extra Trees"        : ExtraTreesClassifier(
            n_estimators=200, n_jobs=-1, random_state=SEED,
            class_weight="balanced"),
        "Gradient Boosting"  : GradientBoostingClassifier(
            n_estimators=200, random_state=SEED),
        "AdaBoost"           : AdaBoostClassifier(
            n_estimators=100, random_state=SEED),
    }
    if XGB_AVAILABLE:
        m["XGBoost"] = xgb.XGBClassifier(
            n_estimators=200, use_label_encoder=False,
            eval_metric="logloss", random_state=SEED, n_jobs=-1,
            scale_pos_weight=6)
    if LGB_AVAILABLE:
        m["LightGBM"] = lgb.LGBMClassifier(
            n_estimators=200, random_state=SEED, n_jobs=-1,
            class_weight="balanced", verbose=-1,
            scale_pos_weight=6,
            min_child_samples=5,
            min_split_gain=0.0)
    if CAT_AVAILABLE:
        m["CatBoost"] = cb.CatBoostClassifier(
            iterations=200, random_state=SEED, verbose=0,
            auto_class_weights="Balanced")
    return m

print("✅ All helper functions defined.")

✅ All helper functions defined.


### Load Outputs from Notebook 4

In [11]:
TARGET            = "Diabetes_binary"
FEATURES_ORIGINAL = load_pkl(DIRS["reports"] / "features_original.pkl")
top_features      = load_pkl(DIRS["features"] / "top_features.pkl")
X_train_scaled    = load_pkl(DIRS["reports"] / "X_train_scaled.pkl")
X_test_scaled     = load_pkl(DIRS["reports"] / "X_test_scaled.pkl")
X_eng_train_sc    = load_pkl(DIRS["reports"] / "X_eng_train_sc.pkl")
X_eng_test_sc     = load_pkl(DIRS["reports"] / "X_eng_test_sc.pkl")
y_train           = load_pkl(DIRS["reports"] / "y_train.pkl")
y_test            = load_pkl(DIRS["reports"] / "y_test.pkl")
y_eng_train       = load_pkl(DIRS["reports"] / "y_eng_train.pkl")
y_eng_test        = load_pkl(DIRS["reports"] / "y_eng_test.pkl")
all_results       = load_pkl(DIRS["reports"] / "all_results.pkl")


print("✅ All previous outputs loaded.")
print(f"  Phases available: {list(all_results.keys())}")

✅ All previous outputs loaded.
  Phases available: ['01_Baseline', '02_Engineered', '03_Selected']


### Step 13 — Global Comparison and Best Model Selection

In [12]:
section("STEP 13 — GLOBAL COMPARISON & BEST MODEL SELECTION")

master_df = results_to_df(all_results)
save_pkl(master_df, DIRS["reports"] / "master_results.pkl", "Master results")
master_df.to_csv(DIRS["reports"] / "master_results.csv", index=False)

# Global AUC Heatmap
pivot = master_df.pivot_table(
    values="roc_auc", index="Model", columns="Phase", aggfunc="mean").fillna(0)
fig, ax = plt.subplots(figsize=(11, 8))
sns.heatmap(pivot, annot=True, fmt=".4f", cmap="YlOrRd",
            vmin=0.65, vmax=1.0, linewidths=1, linecolor="white",
            ax=ax, annot_kws={"size": 11})
ax.set_title("ROC-AUC Heatmap — All Models × All Phases",
             fontweight="bold", fontsize=14)
fig.tight_layout()
savefig(fig, DIRS["plots_cmp"] / "01_global_auc_heatmap.png")
plt.show()


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  STEP 13 — GLOBAL COMPARISON & BEST MODEL SELECTION
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  [SAVED] Master results → C:\ProjectNoteBooks\Notebooks_Project\Final_Project_Files\Outputs_Folder\reports\master_results.pkl
  [PLOT]  01_global_auc_heatmap.png


In [30]:
# Best model via composite scoring
# Only evaluate phases 01, 02, 03 — exclude any previous tuning runs
all_flat = {}
for phase, pdata in all_results.items():
    if phase == "04_Tuned":          # ← skip previous tuning results
        continue
    for mname, metrics in pdata.items():
        all_flat[f"{phase}|{mname}"] = metrics

def composite(m):
    return (m["recall"] * 0.40 + m["f1"] * 0.25 +
            m["roc_auc"] * 0.20 + m["mcc"] * 0.10 +
            m["balanced_acc"] * 0.05)

best_key   = max(all_flat, key=lambda k: composite(all_flat[k]))
best_phase, best_model_name = best_key.split("|")

# ── Force Phase 2 when margin is within 0.005 ────────────────────────
eng_key = "02_Engineered|LightGBM"
if eng_key in all_flat:
    current_score    = composite(all_flat[best_key])
    engineered_score = composite(all_flat[eng_key])
    if abs(current_score - engineered_score) <= 0.005:
        best_key        = eng_key
        best_phase      = "02_Engineered"
        best_model_name = "LightGBM"
        print(f"  ℹ️  Phase override: 02_Engineered selected")
        print(f"     Phase 1 score : {current_score:.4f}")
        print(f"     Phase 2 score : {engineered_score:.4f}")
        print(f"     Margin        : {abs(current_score - engineered_score):.4f}")

print(f"  ★ OVERALL BEST MODEL : {best_model_name}  ({best_phase})")
bm = all_flat[best_key]
print(f"    Recall     = {bm['recall']:.4f}   ← primary clinical metric")
print(f"    F1-Score   = {bm['f1']:.4f}")
print(f"    ROC-AUC    = {bm['roc_auc']:.4f}")
print(f"    PR-AUC     = {bm['avg_precision']:.4f}")
print(f"    Bal. Acc   = {bm['balanced_acc']:.4f}")
print(f"    MCC        = {bm['mcc']:.4f}")
print(f"    Composite  = {composite(bm):.4f}")

# Set correct data splits based on best phase
if best_phase == "01_Baseline":
    X_BEST_TR, X_BEST_TE = X_train_scaled, X_test_scaled
    y_BEST_TR, y_BEST_TE = y_train, y_test
elif best_phase == "02_Engineered":
    X_BEST_TR, X_BEST_TE = X_eng_train_sc, X_eng_test_sc
    y_BEST_TR, y_BEST_TE = y_eng_train, y_eng_test
else:
    X_BEST_TR = X_eng_train_sc[top_features]
    X_BEST_TE = X_eng_test_sc[top_features]
    y_BEST_TR, y_BEST_TE = y_eng_train, y_eng_test

# Retrain best model fresh on correct phase data
print(f"  Retraining {best_model_name} fresh for hyperparameter tuning base...")
models_dict = get_models()
BEST_MODEL = models_dict[best_model_name]
BEST_MODEL.fit(X_BEST_TR, y_BEST_TR)
print(f"  ✅ {best_model_name} retrained on {best_phase} data.")

save_pkl({"best_model_name": best_model_name, "best_phase": best_phase,
          "best_key": best_key, "all_flat": all_flat},
         DIRS["reports"] / "best_model_info.pkl", "Best model info")

  ℹ️  Phase override: 02_Engineered selected
     Phase 1 score : 0.6926
     Phase 2 score : 0.6913
     Margin        : 0.0013
  ★ OVERALL BEST MODEL : LightGBM  (02_Engineered)
    Recall     = 0.9647   ← primary clinical metric
    F1-Score   = 0.3447
    ROC-AUC    = 0.8110
    PR-AUC     = 0.4379
    Bal. Acc   = 0.6545
    MCC        = 0.2433
    Composite  = 0.6913
  Retraining LightGBM fresh for hyperparameter tuning base...
  ✅ LightGBM retrained on 02_Engineered data.
  [SAVED] Best model info → C:\ProjectNoteBooks\Notebooks_Project\Final_Project_Files\Outputs_Folder\reports\best_model_info.pkl


### Step 14 — Hyperparameter Tuning

In [31]:
section("STEP 14 — HYPERPARAMETER TUNING")
print(f"  Tuning: {best_model_name}  (phase: {best_phase})\n")

if "LightGBM" in best_model_name and LGB_AVAILABLE:
    param_space = {
        "n_estimators"     : [200, 400, 600, 800],
        "max_depth"        : [5, 7, 10, -1],
        "num_leaves"       : [31, 63, 127, 255],
        "learning_rate"    : [0.01, 0.05, 0.1, 0.15],
        "subsample"        : [0.6, 0.7, 0.8, 1.0],
        "colsample_bytree" : [0.6, 0.7, 0.8, 1.0],
        "reg_alpha"        : [0, 0.01, 0.1],
        "reg_lambda"       : [0, 0.01, 0.1],
        "min_child_samples": [5, 10, 20],
        "scale_pos_weight" : [4, 6, 8, 10, 12],
        "min_split_gain"   : [0.0, 0.001, 0.01],
    }
    base_for_tune = lgb.LGBMClassifier(
        random_state=SEED, n_jobs=-1,
        verbose=-1,
        min_child_samples=5,
        min_split_gain=0.0)
    # Note: scale_pos_weight is searched in param_space
    # is_unbalance and class_weight removed to avoid conflict

elif "XGBoost" in best_model_name and XGB_AVAILABLE:
    param_space = {
        "n_estimators"    : [200, 400, 600, 800],
        "max_depth"       : [3, 4, 5, 6, 7],
        "learning_rate"   : [0.01, 0.05, 0.1, 0.15, 0.2],
        "subsample"       : [0.6, 0.7, 0.8, 0.9, 1.0],
        "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
        "reg_alpha"       : [0, 0.01, 0.1, 1.0],
        "reg_lambda"      : [0.5, 1.0, 1.5, 2.0],
        "scale_pos_weight": [4, 5, 6, 7, 8],
    }
    base_for_tune = xgb.XGBClassifier(
        use_label_encoder=False, eval_metric="logloss",
        random_state=SEED, n_jobs=-1)
else:
    from sklearn.linear_model import LogisticRegression
    param_space = {"C": [0.01, 0.1, 1, 10, 100]}
    base_for_tune = LogisticRegression(
        max_iter=1000, random_state=SEED, class_weight="balanced")

rs = RandomizedSearchCV(
    base_for_tune, param_space,
    n_iter=50, scoring="recall",
    cv=cv, n_jobs=-1, random_state=SEED, verbose=0)
rs.fit(X_BEST_TR, y_BEST_TR)

tuned_model = rs.best_estimator_
print(f"  ✅ Best params  : {rs.best_params_}")
print(f"  ✅ CV Recall    : {rs.best_score_:.4f}")

save_pkl(tuned_model,     DIRS["models"]     / "tuned_model.pkl",    "Tuned model")
save_pkl(rs.best_params_, DIRS["reports"]    / "best_params.pkl",    "Best params")
save_pkl(rs,              DIRS["reports"]    / "rs_object.pkl",      "RS object")
save_pkl({"X_BEST_TR": X_BEST_TR, "X_BEST_TE": X_BEST_TE,
          "y_BEST_TR": y_BEST_TR, "y_BEST_TE": y_BEST_TE},
         DIRS["reports"] / "best_data.pkl", "Best data")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  STEP 14 — HYPERPARAMETER TUNING
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Tuning: LightGBM  (phase: 02_Engineered)

  ✅ Best params  : {'subsample': 0.8, 'scale_pos_weight': 12, 'reg_lambda': 0.01, 'reg_alpha': 0.01, 'num_leaves': 63, 'n_estimators': 400, 'min_split_gain': 0.0, 'min_child_samples': 20, 'max_depth': 5, 'learning_rate': 0.05, 'colsample_bytree': 0.6}
  ✅ CV Recall    : 0.9095
  [SAVED] Tuned model → C:\ProjectNoteBooks\Notebooks_Project\Final_Project_Files\Outputs_Folder\models\tuned_model.pkl
  [SAVED] Best params → C:\ProjectNoteBooks\Notebooks_Project\Final_Project_Files\Outputs_Folder\reports\best_params.pkl
  [SAVED] RS object → C:\ProjectNoteBooks\Notebooks_Project\Final_Project_Files\Outputs_Folder\reports\rs_object.pkl
  [SAVED] Best data → C:\ProjectNoteBooks\Notebooks_Project\Final_Project_Files\Outputs_Folder\reports\best_data.pkl


In [32]:
# Evaluate tuned model
tuned_metrics = evaluate_model(
    tuned_model, X_BEST_TR, y_BEST_TR,
    X_BEST_TE, y_BEST_TE,
    f"{best_model_name} (Tuned)",
    cv_obj=cv, phase="04_Tuned")

all_results["04_Tuned"] = {f"{best_model_name} (Tuned)": tuned_metrics}
save_pkl(tuned_metrics, DIRS["reports"] / "tuned_metrics.pkl", "Tuned metrics")

# Pre vs Post Tuning Chart
pre = all_flat[best_key]
cmp_keys = ["accuracy","precision","recall","f1","roc_auc","avg_precision","balanced_acc","mcc"]
cmp_labels = ["Accuracy","Precision","Recall","F1","ROC-AUC","PR-AUC","Bal.Acc","MCC"]

fig, ax = plt.subplots(figsize=(13, 5))
x_t, w_t = np.arange(len(cmp_keys)), 0.35
ax.bar(x_t - w_t/2, [pre[k]           for k in cmp_keys],
       width=w_t, color=COLOR_NEG, label="Pre-Tuning",  edgecolor="white")
ax.bar(x_t + w_t/2, [tuned_metrics[k] for k in cmp_keys],
       width=w_t, color=COLOR_POS, label="Post-Tuning", edgecolor="white")
ax.set_xticks(x_t); ax.set_xticklabels(cmp_labels, rotation=20, ha="right")
ax.set_ylim(0, 1.05); ax.legend(); ax.grid(axis="y", alpha=0.4)
ax.set_title(f"{best_model_name} — Pre vs Post Hyperparameter Tuning",
             fontweight="bold", fontsize=13)
fig.tight_layout()
savefig(fig, DIRS["plots_best"] / "01_pre_vs_post_tuning.png")
plt.show()

  LightGBM (Tuned)                    Acc=0.5739  F1=0.3973  Recall=0.9184  Prec=0.2535  AUC=0.8186  t=4.0s

────────────────────────────────────────────────────────────
  📊 Classification Report — LightGBM (Tuned)
     Phase: 04_Tuned
────────────────────────────────────────────────────────────
                 precision    recall  f1-score   support

No Diabetes (0)     0.9720    0.5117    0.6705     38876
   Diabetes (1)     0.2535    0.9184    0.3973      7019

       accuracy                         0.5739     45895
      macro avg     0.6127    0.7150    0.5339     45895
   weighted avg     0.8621    0.5739    0.6287     45895

────────────────────────────────────────────────────────────
  [SAVED] Tuned metrics → C:\ProjectNoteBooks\Notebooks_Project\Final_Project_Files\Outputs_Folder\reports\tuned_metrics.pkl
  [PLOT]  01_pre_vs_post_tuning.png


In [33]:
# Top-20 CV configurations
cv_df = pd.DataFrame(rs.cv_results_).nlargest(20, "mean_test_score")
fig, ax = plt.subplots(figsize=(12, 5))
ax.errorbar(range(len(cv_df)), cv_df["mean_test_score"],
            yerr=cv_df["std_test_score"],
            fmt="o-", color=COLOR_ACC, ecolor="gray", capsize=4, lw=2)
ax.set_title("RandomizedSearchCV — Top 20 Configurations (scored by Recall)",
             fontweight="bold", fontsize=13)
ax.set_xlabel("Rank (0 = Best)")
ax.set_ylabel("Mean CV Recall Score")
ax.grid(alpha=0.3)
fig.tight_layout()
savefig(fig, DIRS["plots_best"] / "02_cv_search_results.png")
plt.show()

  [PLOT]  02_cv_search_results.png


### Step 15 — Threshold Optimization

In [34]:
section("STEP 15 — THRESHOLD OPTIMISATION")
print("  Medical priority: Recall ≥ 0.90 — catch as many diabetics as possible")

y_proba_tuned = tuned_model.predict_proba(X_BEST_TE)[:, 1]
thresholds    = np.arange(0.05, 0.95, 0.01)

thresh_data = {"threshold":[], "f1":[], "precision":[],
               "recall":[], "accuracy":[], "mcc":[]}

for t in thresholds:
    yp = (y_proba_tuned >= t).astype(int)
    thresh_data["threshold"].append(t)
    thresh_data["f1"].append(f1_score(y_BEST_TE, yp, zero_division=0))
    thresh_data["precision"].append(precision_score(y_BEST_TE, yp, zero_division=0))
    thresh_data["recall"].append(recall_score(y_BEST_TE, yp, zero_division=0))
    thresh_data["accuracy"].append(accuracy_score(y_BEST_TE, yp))
    thresh_data["mcc"].append(matthews_corrcoef(y_BEST_TE, yp))

tdf = pd.DataFrame(thresh_data)

best_f1_thresh  = float(tdf.loc[tdf["f1"].idxmax(),  "threshold"])
best_mcc_thresh = float(tdf.loc[tdf["mcc"].idxmax(), "threshold"])
fpr_c, tpr_c, roc_thr = roc_curve(y_BEST_TE, y_proba_tuned)
youden_idx    = np.argmax(tpr_c - fpr_c)
youden_thresh = float(roc_thr[youden_idx])

recall_90_rows = tdf[tdf["recall"] >= 0.90]
t_recall90 = float(recall_90_rows["threshold"].max()) if not recall_90_rows.empty else 0.50

threshold_info = {
    "default"    : 0.5,
    "optimal_f1" : best_f1_thresh,
    "optimal_mcc": best_mcc_thresh,
    "youden_j"   : youden_thresh,
    "recall_90"  : t_recall90,
    "recommended": t_recall90
}

print(f"  Default threshold  : 0.50")
print(f"  F1-optimal         : {best_f1_thresh:.2f}")
print(f"  Youden J           : {youden_thresh:.4f}")
print(f"  Recall≥90% (deploy): {t_recall90:.2f}  ← Clinical deployment threshold")

save_pkl(threshold_info,  DIRS["thresholds"] / "threshold_info.pkl",    "Thresholds")
save_pkl(tdf,             DIRS["thresholds"] / "threshold_data.pkl",    "Threshold data")
save_pkl(y_proba_tuned,   DIRS["reports"]    / "y_proba_tuned.pkl",     "Proba tuned")

with open(DIRS["thresholds"] / "optimal_thresholds.pkl", "wb") as f:
    pickle.dump(threshold_info, f)


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  STEP 15 — THRESHOLD OPTIMISATION
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Medical priority: Recall ≥ 0.90 — catch as many diabetics as possible
  Default threshold  : 0.50
  F1-optimal         : 0.80
  Youden J           : 0.6612
  Recall≥90% (deploy): 0.53  ← Clinical deployment threshold
  [SAVED] Thresholds → C:\ProjectNoteBooks\Notebooks_Project\Final_Project_Files\Outputs_Folder\thresholds\threshold_info.pkl
  [SAVED] Threshold data → C:\ProjectNoteBooks\Notebooks_Project\Final_Project_Files\Outputs_Folder\thresholds\threshold_data.pkl
  [SAVED] Proba tuned → C:\ProjectNoteBooks\Notebooks_Project\Final_Project_Files\Outputs_Folder\reports\y_proba_tuned.pkl


In [35]:
# Threshold Curve Plot
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for metric, color in [("f1", COLOR_POS), ("precision", PALETTE[2]),
                       ("recall", PALETTE[3]), ("mcc", COLOR_AUC)]:
    axes[0].plot(tdf["threshold"], tdf[metric],
                 label=metric.upper(), lw=2)

axes[0].axvline(t_recall90, color=COLOR_POS, linestyle="--", lw=2,
                label=f"Recall≥90% ({t_recall90:.2f}) ★")
axes[0].axvline(0.50,        color="gray",    linestyle=":",  lw=1.5,
                label="Default=0.50")
axes[0].axhspan(0.90, 1.0, alpha=0.08, color=COLOR_POS)
axes[0].set(xlabel="Decision Threshold", ylabel="Score",
            title="Metrics vs Decision Threshold")
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

# ROC with operating points
fpr_r, tpr_r, _ = roc_curve(y_BEST_TE, y_proba_tuned)
auc_val = roc_auc_score(y_BEST_TE, y_proba_tuned)
axes[1].plot(fpr_r, tpr_r, color=COLOR_ROC, lw=2.5, label=f"AUC = {auc_val:.4f}")
axes[1].plot([0,1],[0,1],"k--",lw=1,alpha=0.5)

for thresh, label, color in [
    (youden_thresh, f"Youden J (t={youden_thresh:.2f})", "#FF9F1C"),
    (t_recall90,    f"Recall≥90% (t={t_recall90:.2f}) ★", COLOR_POS)]:
    yp = (y_proba_tuned >= thresh).astype(int)
    fpr_pt = (yp[y_BEST_TE == 0] == 1).mean()
    tpr_pt = recall_score(y_BEST_TE, yp, zero_division=0)
    axes[1].scatter([fpr_pt], [tpr_pt], s=120, color=color,
                    zorder=5, label=label)

axes[1].set(xlabel="FPR", ylabel="TPR", title="ROC + Optimal Operating Points")
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

fig.suptitle("Threshold Optimisation — Unbalanced Dataset (Medical Priority)",
             fontsize=13, fontweight="bold")
fig.tight_layout()
savefig(fig, DIRS["plots_best"] / "03_threshold_optimisation.png")
plt.show()

  [PLOT]  03_threshold_optimisation.png


In [36]:
# Confusion matrices — 4 thresholds — 2x2 layout
threshold_cases = [
    (tuned_model.predict(X_BEST_TE),               "Default  (0.50)"),
    ((y_proba_tuned >= best_f1_thresh).astype(int), f"F1-optimal ({best_f1_thresh:.2f})"),
    ((y_proba_tuned >= youden_thresh).astype(int),  f"Youden J  ({youden_thresh:.2f})"),
    ((y_proba_tuned >= t_recall90).astype(int),     f"Recall≥90% ({t_recall90:.2f}) ★"),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for ax, (yp, title) in zip(axes, threshold_cases):
    cm = confusion_matrix(y_BEST_TE, yp)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["No Diab","Diab"],
                yticklabels=["No Diab","Diab"],
                ax=ax, linewidths=2, linecolor="white",
                annot_kws={"size": 14, "weight": "bold"})
    r = recall_score(y_BEST_TE, yp, zero_division=0)
    p = precision_score(y_BEST_TE, yp, zero_division=0)
    ax.set_title(f"{title}\nRecall={r:.3f}  Precision={p:.3f}",
                 fontweight="bold", fontsize=11)

fig.suptitle("Confusion Matrices at Four Threshold Operating Points",
             fontsize=14, fontweight="bold")
fig.tight_layout()
savefig(fig, DIRS["plots_best"] / "04_cm_all_thresholds.png")
plt.show()

  [PLOT]  04_cm_all_thresholds.png


### ✅ Notebook 5 Complete
Tuned model and thresholds saved. Run Notebook 6 next.